In [63]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from category_encoders import TargetEncoder

import numpy as np
import pandas as pd
from pathlib import Path

In [108]:
file_path = Path.cwd().parent /'data' / 'gandhinagar_property_apartments_recommender_ready.csv'
df = pd.read_csv(file_path)
df.head()   

,location,price_inr_in_lakhs,area_sqft,description,property_url,bathrooms,balconies,current_floor,total_floors,furnishing_status,mapped_area,facing,property_age,property_type,bedrooms,area_type,property_status
0,"Sargasan, Gandhinagar",126.00,2916.0,Experience a new style of living with Altezza ...,https://www.99acres.com/3-bhk-bedroom-apartmen...,3.0,1.0,4.0,8.0,Unknown,Sargasan,Unknown,New,apartment,3.0,Super Built-up,Ready_to_Move
1,"Pethapur, Gandhinagar",51.99,1755.0,Experience a new style of living with Satyamev...,https://www.99acres.com/3-bhk-bedroom-apartmen...,3.0,2.0,3.0,7.0,Unknown,Pethapur,Unknown,New,apartment,3.0,Super Built-up,Under_Construction
2,"Raysan, Gandhinagar",97.00,1908.0,This is your chance to a 3 bhk apartment / fla...,https://www.99acres.com/3-bhk-bedroom-apartmen...,3.0,2.0,4.0,8.0,Unknown,Raysan,Unknown,New,apartment,3.0,Super Built-up,Ready_to_Move
3,"Raysan, Gandhinagar",125.00,2205.0,We are the proud owners of this 3 bhk apartmen...,https://www.99acres.com/3-bhk-bedroom-apartmen...,2.0,2.0,9.0,13.0,Unknown,Raysan,Unknown,New,apartment,3.0,Carpet,Ready_to_Move
4,"Urjanagar 1, Randesan, Gandhinagar",79.00,1755.0,Situated on the top floor (8th) of a mid-Rise ...,https://www.99acres.com/3-bhk-bedroom-apartmen...,3.0,1.0,8.0,13.0,unfurnished,Randesan,Unknown,5-10 years,apartment,3.0,Carpet,Ready_to_Move


In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1304 entries, 0 to 1303
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   location            1304 non-null   str    
 1   price_inr_in_lakhs  1304 non-null   float64
 2   area_sqft           1304 non-null   float64
 3   description         1203 non-null   str    
 4   property_url        1304 non-null   str    
 5   bathrooms           1304 non-null   float64
 6   balconies           1304 non-null   float64
 7   current_floor       1304 non-null   float64
 8   total_floors        1304 non-null   float64
 9   furnishing_status   1304 non-null   str    
 10  mapped_area         1304 non-null   str    
 11  facing              1304 non-null   str    
 12  property_age        1304 non-null   str    
 13  property_type       1304 non-null   str    
 14  bedrooms            1304 non-null   float64
 15  area_type           1304 non-null   str    
 16  property_status  

In [109]:
df.drop(columns=['location','description','property_url'], inplace=True)

In [110]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

In [111]:
numerical = [
    'area_sqft',
    'bathrooms',
    'bedrooms',
    'balconies',
    'current_floor',
    'total_floors'
]

categorical = [
    'mapped_area',
    'facing',
    'property_status',
    'area_type',
    'property_age',
    'property_type'
]
X = df.drop(columns=['price_inr_in_lakhs'])
y = df['price_inr_in_lakhs']
y_transformed = np.log(y)

In [112]:
X = df.drop(columns=['price_inr_in_lakhs'])
y = df['price_inr_in_lakhs']
y_transformed = np.log(y)

In [113]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.35,
    random_state=42
)

In [114]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical),
        ('cat', OneHotEncoder(drop='first',handle_unknown='ignore'), categorical)
    ]
)

In [115]:
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=10))
])

In [116]:
ridge_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [117]:
y_pred_log = ridge_pipeline.predict(X_test)

c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [118]:
print("R2 Score :", r2_score(y_test, y_pred_log))
print("MAE      :", mean_absolute_error(y_test, y_pred_log))
print("RMSE     :", np.sqrt(mean_squared_error(y_test, y_pred_log)))

R2 Score : 0.8384915559271708
MAE      : 0.174254575903011
RMSE     : 0.23893802016601076


In [119]:
actual_price = np.exp(y_test)
predicted_price = np.exp(y_pred_log)
print("Original Scale Metrics")

print(
    "MAE:",
    mean_absolute_error(actual_price, predicted_price)
)

print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(
            actual_price,
            predicted_price
        )
    )
)

Original Scale Metrics
MAE: 19.260827504336003
RMSE: 49.152877547837335


In [120]:
feature_names = (
    ridge_pipeline
    .named_steps['preprocessor']
    .get_feature_names_out()
)

coefficients = (
    ridge_pipeline
    .named_steps['model']
    .coef_
)

In [121]:
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

importance_df['Importance'] = (
    importance_df['Coefficient']
    .abs()
)

importance_df = importance_df.sort_values(
    'Importance',
    ascending=False
)

In [101]:
# importance_df.head(100)

In [102]:
scaler.scale_

array([8.58654964e+02, 7.98945373e-01, 7.67911927e-01, 6.83369900e-01,
       3.53345300e+00, 6.49871236e+00])

In [122]:
std_dict = {
    'num__area_sqft': 8.94141235e+02,
    'num__bathrooms': 8.03937686e-01,
    'num__bedrooms': 7.73154981e-01,
    'num__balconies': 6.88713917e-01,
    'num__current_floor': 3.51038922e+00,
    'num__total_floors': 6.52079505e+00
}

In [123]:
def get_percentage_impact(row):

    coef = row['Coefficient']
    feature = row['Feature']

    # Numerical features (scaled)
    if feature in std_dict:
        return (np.exp(coef / std_dict[feature]) - 1) * 100

    # Categorical features (not scaled)
    else:
        return (np.exp(coef) - 1) * 100

In [124]:
importance_df['Percentage_Impact'] = (
    importance_df.apply(get_percentage_impact, axis=1)
)

importance_df['Percentage_Impact'] = (
    importance_df['Percentage_Impact'].round(2)
)

In [125]:
# importance_df[['Feature','Coefficient','Percentage_Impact']]
display_df = importance_df[
    ['Feature','Coefficient','Percentage_Impact']
].copy()

display_df['Interpretation'] = (
    display_df['Percentage_Impact']
    .apply(lambda x: f"{x:+.2f}%")
)

In [126]:
display_df.head(1000)

,Feature,Coefficient,Percentage_Impact,Interpretation
9,cat__mapped_area_Gift City,0.490915,63.38,+63.38%
10,cat__mapped_area_Kalol,-0.443123,-35.80,-35.80%
26,cat__mapped_area_Sector 22,0.217694,24.32,+24.32%
2,num__bedrooms,0.211194,31.41,+31.41%
0,num__area_sqft,0.187909,0.02,+0.02%
17,cat__mapped_area_Pethapur,-0.181874,-16.63,-16.63%
20,cat__mapped_area_Raysan,0.166874,18.16,+18.16%
30,cat__mapped_area_Sector 6,0.162748,17.67,+17.67%
19,cat__mapped_area_Randheja,-0.151450,-14.05,-14.05%
5,num__total_floors,0.119935,1.86,+1.86%


In [127]:
intercept = ridge_pipeline.named_steps['model'].intercept_
print(intercept)

4.3257735813878195


In [ ]:
display_df.to_csv(
    Path.cwd().parent / 'data' / 'gandhinagar_property_apartments_model_insights.csv',
    index=False
)

In [ ]:
ohe = ridge_pipeline.named_steps['preprocessor'].named_transformers_['cat']
mapped_area_categories = ohe.categories_[0]
print(mapped_area_categories)

c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


R2 Score : 0.8399542494556664
MAE      : 0.17351646502903087
RMSE     : 0.23785359392046196
Original Scale Metrics
MAE: 19.071534762887374
RMSE: 47.868980775845046


NameError: name 'scaler' is not defined